# Exploratory Data Analysis (EDA) - SMS Spam Detection

This notebook analyzes the **SMS Spam Collection** dataset used in the FreeCodeCamp Machine Learning certification.
We investigate class distribution, text length statistics, token distributions, and data leakage.

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from collections import Counter

import config
import data

# Ensure data is downloaded and loaded
train_df, valid_df = data.load_raw_data()
print(f"Raw Train shape: {train_df.shape}")
print(f"Raw Valid shape: {valid_df.shape}")

## 1. Class Distribution Analysis

Let's inspect the balance between 'ham' (legitimate) and 'spam' messages.

In [ ]:
df_all = pd.concat([train_df, valid_df], ignore_index=True)
counts = df_all['label'].value_counts()
percentages = df_all['label'].value_counts(normalize=True) * 100

print(pd.DataFrame({'Count': counts, 'Percentage (%)': percentages}))

plt.figure(figsize=(6, 4))
bars = plt.bar(counts.index, counts.values, color=['#2ecc71', '#e74c3c'], width=0.5)
plt.title('SMS Dataset Class Distribution (Severe Imbalance)', fontsize=12, fontweight='bold')
plt.ylabel('Message Count')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 50, f'{yval:,} ({yval/len(df_all)*100:.1f}%)', ha='center', va='bottom')
plt.tight_layout()
plt.show()

## 2. Message Length Distribution (Ham vs Spam)

Spammers typically maximize the 160-character SMS limit with dense call-to-actions, whereas normal messages vary widely.

In [ ]:
df_all['char_length'] = df_all['message'].apply(len)
df_all['word_count'] = df_all['message'].apply(lambda s: len(s.split()))

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.hist(df_all[df_all['label'] == 'ham']['char_length'], bins=50, alpha=0.6, color='green', label='Ham', density=True)
plt.hist(df_all[df_all['label'] == 'spam']['char_length'], bins=50, alpha=0.6, color='red', label='Spam', density=True)
plt.title('Character Length Distribution')
plt.xlabel('Number of Characters')
plt.ylabel('Density')
plt.legend()

plt.subplot(1, 2, 2)
plt.hist(df_all[df_all['label'] == 'ham']['word_count'], bins=30, alpha=0.6, color='green', label='Ham', density=True)
plt.hist(df_all[df_all['label'] == 'spam']['word_count'], bins=30, alpha=0.6, color='red', label='Spam', density=True)
plt.title('Word Count Distribution')
plt.xlabel('Number of Words')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.show()

## 3. Duplicate and Data Leakage Audit

In [ ]:
leakage_info = data.inspect_leakage_and_duplicates(train_df, valid_df)
for k, v in leakage_info.items():
    print(f"{k}: {v}")

## 4. Top Distinctive Keywords per Class

In [ ]:
df_all['cleaned'] = df_all['message'].apply(data.clean_text)

spam_words = ' '.join(df_all[df_all['label'] == 'spam']['cleaned']).split()
ham_words = ' '.join(df_all[df_all['label'] == 'ham']['cleaned']).split()

spam_counter = Counter(spam_words).most_common(15)
ham_counter = Counter(ham_words).most_common(15)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
s_words, s_counts = zip(*spam_counter)
axes[0].barh(s_words[::-1], s_counts[::-1], color='#e74c3c')
axes[0].set_title('Top Words in SPAM Messages')

h_words, h_counts = zip(*ham_counter)
axes[1].barh(h_words[::-1], h_counts[::-1], color='#2ecc71')
axes[1].set_title('Top Words in HAM Messages')
plt.tight_layout()
plt.show()